# Module 5: Neural networks, by intuition

The worked solution. Try the build in `notebooks/` first; getting stuck for a while teaches more than reading this. The last cells check every done-when criterion.

In [ ]:
# Runs only on Google Colab: fetch the starter kit and its packages, and copy any keys
# you saved in Colab's Secrets panel into the environment. Locally this does nothing.
import os, sys
if "google.colab" in sys.modules:
    !git clone --depth 1 https://github.com/Go-Everyway/ai-track-starter /content/ai-track-starter
    %cd /content/ai-track-starter
    %pip install -q -r requirements.txt
    from google.colab import userdata
    for name in ["LLM_API_KEY", "LLM_PROVIDER", "LLM_BASE_URL", "LLM_MODEL", "EMBED_MODEL", "LLM_DAILY_BUDGET_USD"]:
        try:
            os.environ[name] = userdata.get(name)
        except Exception:
            pass  # not set in Secrets; lib/llm.py uses its default

In [ ]:
# Find the repo root so paths work from notebooks/ or from Colab.
import sys
from pathlib import Path
import pandas as pd

ROOT = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / "data" / "leads_clean.csv").exists())
DATA = ROOT / "data"
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))
print("starter kit at", ROOT)

## 1. Look at the companies

400 synthetic company profiles. 40 are marked `is_customer`: Tallyhawk's best accounts.
The job: given any company, find the ones most like it, and use that to find prospects
that look like the customers.

In [ ]:
import numpy as np
from lib import llm

companies = pd.read_json(DATA / "companies.jsonl", lines=True)
print(f"{len(companies)} companies, {companies.is_customer.sum()} customers")
print("embedding mode:", "mock (word-hash vectors, mechanics only)" if llm._settings()["provider"] == "mock" else llm._settings()["embed_model"])
companies[["company_id", "name", "industry", "employees", "description"]].head()

## 2. Cosine similarity by hand

Three tiny made-up vectors. Cosine similarity is the dot product divided by the two lengths.
1 means pointing the same way, 0 means unrelated.

In [ ]:
a = np.array([3.0, 1.0, 0.0])   # say: a distributor
b = np.array([2.0, 1.0, 0.0])   # another distributor
c = np.array([0.0, 1.0, 3.0])   # a dental clinic

def cos(u, v):
    return float(u @ v / (np.linalg.norm(u) * np.linalg.norm(v)))

print(f"a vs b: {cos(a, b):.3f}")
print(f"a vs c: {cos(a, c):.3f}")
print(f"b vs c: {cos(b, c):.3f}")

## 3. Embed every description, once

One call per batch of 50. Results are cached in `logs/`, so rerunning the notebook doesn't
pay again. The cache key includes the model, so switching models re-embeds.

In [ ]:
import hashlib, json

def embed_cached(texts, batch=50):
    s = llm._settings()
    model = "mock" if s["provider"] == "mock" else s["embed_model"]
    path = ROOT / "logs" / f"embeddings-{hashlib.md5(model.encode()).hexdigest()[:10]}.json"
    cache = json.loads(path.read_text()) if path.exists() else {}
    key = lambda t: hashlib.md5(t.encode()).hexdigest()
    todo = [t for t in dict.fromkeys(texts) if key(t) not in cache]
    for i in range(0, len(todo), batch):
        chunk = todo[i:i + batch]
        for t, v in zip(chunk, llm.embed(chunk)):
            cache[key(t)] = v
    path.parent.mkdir(exist_ok=True)
    path.write_text(json.dumps(cache))
    return np.array([cache[key(t)] for t in texts])

E = embed_cached(companies.description.tolist())
E = E / np.linalg.norm(E, axis=1, keepdims=True)   # unit length, so a dot product is the cosine
print("embeddings:", E.shape)

## OPERATOR: change one value

`OPERATOR_VALUE` is how many neighbours the lookup returns for each company. The default is 10.
Try anything from 3 to 25. Fewer neighbours means only the very closest matches; more means
the list reaches further out. Watch the lookup table and the same-industry comparison below:
they both change.

In [ ]:
OPERATOR_VALUE = 10  # change this, then Runtime → Run all
DEFAULT_VALUE = 10
K = int(OPERATOR_VALUE)
print(f"Showing {K} neighbours per company.")

## 4. The lookup: accounts like this one

In [ ]:
def lookalikes(company_id, vectors=E, k=K):
    """The k companies most similar to `company_id`, best first, excluding itself."""
    i = companies.index[companies.company_id == company_id][0]
    sims = vectors @ vectors[i]
    sims[i] = -np.inf
    top = np.argsort(-sims)[:k]
    out = companies.iloc[top][["company_id", "name", "industry", "employees", "is_customer"]].copy()
    out["similarity"] = sims[top].round(3)
    return out.reset_index(drop=True)

example = companies[companies.is_customer].company_id.iloc[0]
print(companies.set_index("company_id").loc[example, "description"], "\n")
lookalikes(example)

## 5. Prospects that look like your best customers

Score each non-customer by its similarity to the closest customer, then rank.

In [ ]:
cust = companies.is_customer.to_numpy()
best_match = (E[~cust] @ E[cust].T).max(axis=1)
prospects = companies[~cust][["company_id", "name", "industry", "employees"]].copy()
prospects["closest_customer_sim"] = best_match.round(3)
prospects = prospects.sort_values("closest_customer_sim", ascending=False).reset_index(drop=True)
prospects.head(10)

## 6. Judge the matches

Read the neighbours for a few companies. A good match is a company a salesperson would
call "like this one": similar business, similar size, similar situation. Record two bad
matches and why the embedding put them there.

In [ ]:
for cid in companies.company_id.sample(3, random_state=1):
    row = companies.set_index("company_id").loc[cid]
    print(f"\n=== {row['name']} ({row.industry}, {row.employees} employees)")
    print(lookalikes(cid, k=5)[["name", "industry", "employees", "similarity"]].to_string(index=False))

In [ ]:
# From the mock-mode run. With a real embedding model your bad matches will differ; write down yours.
bad_matches = {
    "Ulahaven Supply -> Marurford Care": "An auto-parts distributor's closest match is a healthcare "
        "company. Both descriptions say mid-size, cloud ERP and a similar headcount, and those shared "
        "template words outweighed what each company actually does.",
    "Halacore Wholesale -> Tamemere Academy": "A wholesaler matched a training company. The size and "
        "ERP sentences are worded the same, and mock vectors weigh every word equally, so the one "
        "word that matters (wholesale versus education) barely moves the score.",
}

## 7. Compare with keyword search

TF-IDF scores words by how distinctive they are. It's the classic keyword baseline.
One rough check for both: of each company's 10 neighbours, what share are in the same industry?

In [ ]:
from sklearn.feature_extraction.text import TfidfVectorizer

T = TfidfVectorizer(stop_words="english").fit_transform(companies.description).toarray()
T = T / np.linalg.norm(T, axis=1, keepdims=True)

def same_industry_rate(vectors, k=K):
    sims = vectors @ vectors.T
    np.fill_diagonal(sims, -np.inf)
    top = np.argsort(-sims, axis=1)[:, :k]
    ind = companies.industry.to_numpy()
    return float((ind[top] == ind[:, None]).mean())

random_rate = float((companies.industry.value_counts(normalize=True) ** 2).sum())
comparison = pd.Series({
    "embeddings": same_industry_rate(E),
    "TF-IDF keywords": same_industry_rate(T),
    "random neighbours": random_rate,
}).round(3)
comparison

In [ ]:
keyword_wins = """
Keyword search wins when the match has to be exact: a product name, a part number, an ERP
the prospect runs, or a specific signal like "hiring a demand planner". Embeddings blur
those into general meaning, so a search for one named system can return companies that
merely sound similar.
"""

## Done-when checks

In [ ]:
from lib.checks import done_when

def lookup_works(cid):
    r = lookalikes(cid)
    return (len(r) == K and cid not in set(r.company_id)
            and r.similarity.notna().all() and r.similarity.is_monotonic_decreasing)

from lib.figures import record

checks = [
    ("The lookup returns ranked neighbours for any company",
     all(lookup_works(c) for c in companies.company_id.sample(5, random_state=0))),
    ("You compared embeddings with a keyword baseline", {"embeddings", "TF-IDF keywords"} <= set(comparison.index)),
    ("You explained at least two bad matches",
     len(bad_matches) >= 2 and all(len(v.split()) >= 8 for v in bad_matches.values())),
    ("You wrote down when keyword search would do better", len(keyword_wins.split()) >= 20),
]

if OPERATOR_VALUE == DEFAULT_VALUE:
    mode = llm.settings()["provider"]
    record("m05", mode=mode, companies=len(companies), customers=int(companies.is_customer.sum()),
           cosine_a_b=round(cos(a, b), 3), cosine_a_c=round(cos(a, c), 3), cosine_b_c=round(cos(b, c), 3),
           same_industry_embeddings=float(comparison["embeddings"]),
           same_industry_tfidf=float(comparison["TF-IDF keywords"]),
           same_industry_random=float(comparison["random neighbours"]), neighbours=K)
    assert done_when(checks)
else:
    done_when(checks)
    print(f"\n(OPERATOR_VALUE is {OPERATOR_VALUE}, not the default {DEFAULT_VALUE}, so this is a check, not a test.)")